In [1]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm
from scipy.io import loadmat
from scipy.optimize import curve_fit
import os

%matplotlib inline
%config InlineBackend.figure_formats = ['svg']

## Load dataset

In [2]:
dat = loadmat('data_final/dataset_Mar23_2026.mat')

locals().update(dat) #Populates all variables: ek, chi, rac, mc, omc

ek = ek.flatten()
chi = chi.flatten()

## Perform fit

In [3]:
def get_fit(ek,Rc,R1,R_tilde):
    return (Rc + ek**(2./9.) * R1 + ek**(1./6.) * R_tilde)

Rc      = np.zeros_like(chi)
R1      = np.zeros_like(chi)
R_tilde = np.zeros_like(chi)

ekMask = ek < 1e-4 #Fit only for Ekman numbers <= 1e-5

for k, chiComp in enumerate(chi):
    ek_curve_fit = ek[ekMask]
    R_curve_fit = rac[ekMask,k] * ek[ekMask]**(4./3.)
    [Rc[k],R1[k],R_tilde[k]], pcov = curve_fit(get_fit,ek_curve_fit,R_curve_fit,
                                               p0=[0.2, 0.2, 0.2],
                                               bounds=([-np.inf, -np.inf, -np.inf],
                                                       [np.inf,np.inf,np.inf]))

chiMask = chi > 0.2 #Fit only for chi > 0.2
pc = np.polyfit(chi[chiMask],np.log10(Rc[chiMask]),deg=5)
p1 = np.polyfit(chi[chiMask],np.log10(R1[chiMask]),deg=5)
p_tilde = np.polyfit(chi[chiMask],np.log10(np.abs(R_tilde[chiMask])),deg=5)

## Extrapolate to any value of $\chi > 0.2$ and $E$

In [4]:
def get_rac(ek, chi):
    RcFit = 10**np.polyval(pc,chi)
    R1Fit = 10**np.polyval(p1,chi)
    R_tildeFit = -10**np.polyval(p_tilde,chi)
    Rac_Fit = get_fit(ek,RcFit,R1Fit,R_tildeFit)
    return Rac_Fit * ek**(-4./3.)

In [11]:
ekFit = np.array([1e-11,1e-15])
chiFit = 0.5

rac_fit = get_rac(ekFit,chiFit)
print(rac_fit)

[6.14629607e+14 1.32728638e+20]
